# Fit C: the estimated ceilings are imposed, not learned, and axis 2 is three basins
Only **3 of 98** benchmarks pull the estimated gap off its Beta(1, 49) prior, all three saturated OLD items (SWE-Bench Verified, WMDP Biology, MMLU); the two fixed FrontierMath v1 walls sit **0.046 / 0.121** above the highest score ever recorded, so the data could not have found them.
Six chains split **three** ways at alignment 0.95 — {0,2,3} / {5} / {1,4} — on whether GBAEval loads on commonsense or on the fluid axis; the leading basin alone is converged (eta r̂ **1.025**), the 0.90 merge that swallows chain 5 is not (**1.473**).
Model ordering survives it: Spearman **+0.983** on the summed axes and **+0.967** on hard math, but only **+0.700** on axis 2.

In [1]:
import sys, json, itertools
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "fit.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import xarray as xr
import arviz as az
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.special import expit
from scipy.stats import beta as beta_dist, spearmanr

from config import ECI_EPS
from data import load_eci_data, load_benchmark_floors, clip_scores_to_floors
from analysis import mirt_identified_rhat
from viz import benchmark_icc_fig

TRACE = ROOT / ("results/mirt_humanprior_lineageprior_lineagebm_floors_ceilings_ceilnoise"
                "/trace_mirt_k3_humanprior_lineageprior_lineagebm_floors_ceilings_ceilnoise.nc")
PLOTS = ROOT / "plots/k3_c_ceilings_axis2"
PLOTS.mkdir(parents=True, exist_ok=True)
THIN = 5

C = dict(blue="#0072B2", sky="#56B4E9", orange="#E69F00", verm="#D55E00",
         green="#009E73", pink="#CC79A7", gray="#999999", dark="#333333")

def show(fig, name):
    fig.update_layout(template="plotly_white", title=dict(x=0.5, font=dict(size=13)))
    fig.write_image(PLOTS / f"{name}.png", scale=2)
    fig.show()

# ── data: exactly what the fit saw (floors clipped, then Beta boundary clip) ──
data = clip_scores_to_floors(load_eci_data(include_all_benchmarks=True),
                             load_benchmark_floors(load_eci_data(include_all_benchmarks=True)))
FLOOR = load_benchmark_floors(data)
BEN = data.blookup.sort_values("benchmark_idx")["benchmark"].tolist()
MOD = data.mlookup.sort_values("model_idx")["model"].tolist()
YOBS = np.clip(data.scores, ECI_EPS, 1.0 - ECI_EPS)
NB, NM = len(BEN), len(MOD)
CAT = (pd.read_csv(ROOT / "1_data/processed/benchmarks_merged.csv")
       .drop_duplicates("benchmark").set_index("benchmark")["category"])
NOBS_B = np.array([int((data.bench_idx == i).sum()) for i in range(NB)])
MAXOBS = np.array([YOBS[data.bench_idx == i].max() for i in range(NB)])
assert (len(data.scores), NM, NB) == (4445, 765, 98)

# ── trace: opened once, thinned; log_likelihood only in the LOO cell ─────────
post = xr.open_dataset(TRACE, group="posterior")[
    ["A", "theta", "D", "ceiling_gap", "ceiling_d"]].isel(draw=slice(None, None, THIN)).load()
stats = xr.open_dataset(TRACE, group="sample_stats")[["logp", "diverging"]].load()
NC = post.sizes["chain"]
LP = stats["logp"].values.mean(axis=1)
LPQ = np.quantile(stats["logp"].values, [0.05, 0.95], axis=1)
DIV = stats["diverging"].values.sum(axis=1)
GOF_FILE = json.loads((TRACE.parent / "mirt_gof_k3.json").read_text())

# ── basins: cluster chains on best-permutation loading correlation ───────────
PERMS = list(itertools.permutations(range(3)))
Amed = np.median(post["A"].values, axis=1)          # (chain, bench, latent)
THmed = np.median(post["theta"].values, axis=1)     # (chain, model, latent)

def match(c1, c2):
    """Best axis permutation of chain c2 onto c1, with matched per-axis corrs."""
    M = np.corrcoef(Amed[c1].T, Amed[c2].T)[:3, 3:]
    p = max(PERMS, key=lambda q: sum(M[i, q[i]] for i in range(3)))
    return list(p), np.array([M[i, p[i]] for i in range(3)])

# alignment score = WORST matched axis. 0.95 splits 3 ways, 0.90 only 2.
ALIGN = np.array([[match(a, b)[1].min() for b in range(NC)] for a in range(NC)])

def cluster(thr):
    lab = -np.ones(NC, int)
    for c in range(NC):
        if lab[c] < 0:
            lab[c] = lab.max() + 1
            lab[(lab < 0) & (ALIGN[c] >= thr)] = lab[c]
    order = sorted(range(lab.max() + 1), key=lambda g: -LP[lab == g].mean())
    return [[c for c in range(NC) if lab[c] == g] for g in order]

G95, G90 = cluster(0.95), cluster(0.90)
assert G95 == [[0, 2, 3], [5], [1, 4]], G95
assert G90 == [[0, 2, 3, 5], [1, 4]], G90
LEAD, CH5, ALT = G95                       # leading / chain-5 / alternative basin
BNAME = {tuple(LEAD): "basin A · chains 0,2,3", tuple(CH5): "basin B · chain 5",
         tuple(ALT): "basin C · chains 1,4"}
BCOL = {tuple(LEAD): C["blue"], tuple(CH5): C["sky"], tuple(ALT): C["verm"]}

# every chain permuted onto the first leading-basin chain: one common frame
REF = LEAD[0]
PERM = [match(REF, c)[0] for c in range(NC)]
def bmean(med, g): return np.mean([med[c][:, PERM[c]] for c in g], axis=0)
AB = {tuple(g): bmean(Amed, g) for g in G95}          # loadings per basin
TB = {tuple(g): bmean(THmed, g) for g in G95}         # abilities per basin
AB90 = {tuple(g): bmean(Amed, g) for g in G90}
TB90 = {tuple(g): bmean(THmed, g) for g in G90}

# ── axis naming: mean loading of probe bundles, never the top-loading item ───
CLASSIC = ["GSM8K", "OpenBookQA", "ARC (AI2)", "PIQA", "HellaSwag",
           "Adversarial NLI", "CSQA2"]
PROBE = {"Hard math + science": ["FrontierMath Tier 4", "OTIS Mock AIME 2024-2025",
                                 "MATH Level 5", "FrontierMath"],
         "Easy knowledge / commonsense": CLASSIC,
         "Fluid / abstract": ["VPCT", "ARC-AGI-2", "ARC-AGI"]}
PIDX = {k: [BEN.index(b) for b in v] for k, v in PROBE.items()}
CI, GBI = PIDX["Easy knowledge / commonsense"], BEN.index("GBAEval")
Pm = np.array([AB[tuple(LEAD)][PIDX[k]].mean(axis=0) for k in PROBE])
q = max(PERMS, key=lambda p: sum(Pm[i, p[i]] for i in range(3)))
assert all(int(np.argmax(Pm[i])) == q[i] for i in range(3)), "probes not separable"
AXIS = [None] * 3
for i, k in enumerate(PROBE):
    AXIS[q[i]] = k
K2 = AXIS.index("Easy knowledge / commonsense")       # "axis 2" everywhere below
assert (K2 == 1) and AXIS[0].startswith("Hard"), AXIS
AXCOL = [C["green"], C["orange"], C["pink"]]

# ── ceilings. delta reaches the likelihood only through log(d_b - c_b), which
#    carries no axis label, so the gap is read pooled over all six chains.
FIXED_D = np.array(json.loads(post.attrs["mirt_fixed_ceiling_d"]))
CAPPED = [BEN[i] for i in np.flatnonzero(FIXED_D < 1.0)]
assert sorted(CAPPED) == ["FrontierMath Tier 4 v1", "FrontierMath v1"], CAPPED
GAP = post["ceiling_gap"].values.reshape(-1, NB)
CD = post["ceiling_d"].values.reshape(-1, NB)
GMED, GSD = np.median(GAP, axis=0), GAP.std(axis=0)
GQ = np.quantile(GAP, [0.05, 0.95], axis=0)
DMEAN = CD.mean(axis=0)
HEAD = DMEAN - MAXOBS                                  # headroom left under d
APPR = (MAXOBS - FLOOR) / (DMEAN - FLOOR)              # 1.0 = obs press the wall
PRI = beta_dist(1.0, 49.0)                             # the ceiling_noise prior
P_MED, P_SD, P_Q95 = PRI.median(), PRI.std(), PRI.ppf(0.95)
OFF = GMED > P_Q95                                     # moved off the prior
GAP_B = [np.median(GAP.reshape(NC, -1, NB)[g].reshape(-1, NB), axis=0) for g in G95]

# ── fitted values, same link as the fit, per chain subset ───────────────────
def fit_mu(sel, cap=200):
    idx = np.linspace(0, post.sizes["draw"] - 1, min(cap, post.sizes["draw"])).astype(int)
    sub = post.isel(chain=sel, draw=idx)
    Av = sub["A"].values.reshape(-1, NB, 3)
    Tv = sub["theta"].values.reshape(-1, NM, 3)
    eta = -sub["D"].values.reshape(-1, NB)[:, data.bench_idx]
    for k in range(3):
        eta += Av[:, data.bench_idx, k] * Tv[:, data.model_idx, k]
    c = FLOOR[data.bench_idx]
    d = sub["ceiling_d"].values.reshape(-1, NB)[:, data.bench_idx]
    return (c + (d - c) * expit(eta)).mean(axis=0)

def gof(mu):
    r = mu - YOBS
    return dict(rmse=float(np.sqrt((r ** 2).mean())), mae=float(np.abs(r).mean()),
                r2=float(1 - (r ** 2).sum() / ((YOBS - YOBS.mean()) ** 2).sum()))

GF = {"pooled (6 chains)": gof(fit_mu(list(range(NC))))}
for g in G95:
    GF[BNAME[tuple(g)]] = gof(fit_mu(g))
GF["0.90 merge · 0,2,3,5"] = gof(fit_mu(G90[0]))

# ── r-hat and ESS are thinning-sensitive by construction, so they are the one
#    quantity read off EVERY draw; the arrays are released straight after.
full = xr.open_dataset(TRACE, group="posterior")[
    ["A", "theta", "D", "sigma_b", "tau_CD"]].load()
SETS = {"pooled (6 chains)": list(range(NC)), "0.90 merge · 0,2,3,5": G90[0],
        BNAME[tuple(LEAD)]: LEAD, BNAME[tuple(ALT)]: ALT}
RH, ESS = {}, {}
for nm, sel in SETS.items():
    RH[nm] = mirt_identified_rhat(az.InferenceData(posterior=full.isel(chain=sel)), data)
    e = az.ess(full.isel(chain=sel), var_names=["D"])["D"].values
    ESS[nm] = (float(e.min()), float(np.median(e)))
del full

# ── assertions against the validated readouts ───────────────────────────────
VAL_ALIGN = np.array([[1.00, .60, 1.00, 1.00, .60, .91], [.60, 1.00, .60, .60, 1.00, .57],
                      [1.00, .60, 1.00, 1.00, .60, .91], [1.00, .60, 1.00, 1.00, .60, .91],
                      [.60, 1.00, .60, .60, 1.00, .56], [.91, .57, .91, .91, .56, 1.00]])
assert np.abs(ALIGN - VAL_ALIGN).max() < 0.005, np.round(ALIGN, 3)
assert np.abs(LP - np.array([3441.8, 3427.6, 3438.4, 3452.4, 3433.7, 3441.7])).max() < 0.1
for nm, (eta, dd, emin, emed) in {
        "pooled (6 chains)": (1.549, 1.631, 10, 37),
        "0.90 merge · 0,2,3,5": (1.473, 1.369, 9, 408),
        BNAME[tuple(ALT)]: (1.056, 1.048, 41, 990)}.items():
    assert abs(RH[nm]["eta_max_rhat"] - eta) < 0.002, (nm, RH[nm]["eta_max_rhat"])
    assert abs(RH[nm]["D_max_rhat"] - dd) < 0.002, (nm, RH[nm]["D_max_rhat"])
    assert (round(ESS[nm][0]), round(ESS[nm][1])) == (emin, emed), (nm, ESS[nm])
for nm, (rmse, mae, r2) in {"0.90 merge · 0,2,3,5": (0.0461, 0.0307, 0.9704),
                            BNAME[tuple(ALT)]: (0.0467, 0.0311, 0.9696)}.items():
    g_ = GF[nm]
    assert (abs(g_["rmse"] - rmse) < 5e-5 and abs(g_["mae"] - mae) < 5e-5
            and abs(g_["r2"] - r2) < 5e-4), (nm, g_)
assert [BEN[i] for i in np.flatnonzero(GMED > 0.10)] == ["MMLU", "SWE-Bench Verified",
                                                         "WMDP Biology"]
assert np.abs(np.array([GMED[BEN.index(b)] for b in
                        ["SWE-Bench Verified", "WMDP Biology", "MMLU"]])
              - np.array([0.175, 0.159, 0.140])).max() < 0.002

print(f"{len(data.scores)} obs · {NM} models · {NB} benchmarks · "
      f"{NC} chains x {post.sizes['draw']} thinned draws (THIN={THIN})")
print(f"axes: {AXIS} · axis 2 = column {K2}")
print(f"basins @0.95: {G95} logp {[round(float(LP[g].mean()), 1) for g in G95]} · "
      f"@0.90: {G90} · divergences per chain {list(map(int, DIV))}")
print(f"fixed ceilings: {[(b, float(FIXED_D[BEN.index(b)])) for b in sorted(CAPPED)]} · "
      f"{int(OFF.sum())} of {NB} benchmarks pull the gap past the prior 95th pct "
      f"({P_Q95:.3f}); {int((GMED > 0.10).sum())} past 0.10")
print(f"gap is basin-invariant: corr(median) {np.corrcoef(GAP_B[0], GAP_B[2])[0, 1]:.4f}, "
      f"largest per-benchmark disagreement {np.abs(GAP_B[0] - GAP_B[2]).max():.4f}")
print("every validated number asserted OK")

4445 obs · 765 models · 98 benchmarks · 6 chains x 400 thinned draws (THIN=5)
axes: ['Hard math + science', 'Easy knowledge / commonsense', 'Fluid / abstract'] · axis 2 = column 1
basins @0.95: [[0, 2, 3], [5], [1, 4]] logp [3444.2, 3441.7, 3430.6] · @0.90: [[0, 2, 3, 5], [1, 4]] · divergences per chain [0, 18, 0, 0, 0, 0]
fixed ceilings: [('FrontierMath Tier 4 v1', 0.6), ('FrontierMath v1', 0.57)] · 6 of 98 benchmarks pull the gap past the prior 95th pct (0.059); 3 past 0.10
gap is basin-invariant: corr(median) 0.9895, largest per-benchmark disagreement 0.0194
every validated number asserted OK


### 1 · The three benchmarks with a large estimated gap have their top scores sitting **on** the fitted asymptote; the two fixed FrontierMath v1 walls have nothing near theirs.
The repo's own ICC, μ = c + (d − c)·σ(η), on eight benchmarks: the 3 large-gap movers, the 2 fixed-d items, and 3 controls whose gap is **≤ 0.006**. η is the draw-mean A·θ − D of the leading basin, d the posterior-mean `ceiling_d`. Dropdown opens on MMLU.

In [2]:
PICK = ["MMLU", "SWE-Bench Verified", "WMDP Biology",              # gap > 0.10
        "FrontierMath v1", "FrontierMath Tier 4 v1",                # fixed d < 1
        "ARC-AGI-2", "MATH Level 5", "OTIS Mock AIME 2024-2025"]    # controls, gap <= 0.006
assert max(GMED[BEN.index(b)] for b in PICK[5:]) < 0.007

# eta averaged over draws, not built from marginal means: it is the linear
# predictor the likelihood saw, and it needs no axis permutation.
sub = post.isel(chain=LEAD, draw=np.linspace(0, post.sizes["draw"] - 1, 200).astype(int))
Av = sub["A"].values.reshape(-1, NB, 3)
Tv = sub["theta"].values.reshape(-1, NM, 3)
ETA = -sub["D"].values.reshape(-1, NB)[:, data.bench_idx]
for k in range(3):
    ETA += Av[:, data.bench_idx, k] * Tv[:, data.model_idx, k]
ETA = ETA.mean(axis=0)
D_LEAD = post["ceiling_d"].isel(chain=LEAD).values.reshape(-1, NB).mean(axis=0)

keep = np.flatnonzero(np.isin([BEN[i] for i in data.bench_idx], PICK))
bench_of = [BEN[i] for i in data.bench_idx[keep]]
fig = benchmark_icc_fig(ETA[keep], YOBS[keep], [MOD[i] for i in data.model_idx[keep]],
                        bench_of, floor={b: float(FLOOR[BEN.index(b)]) for b in PICK},
                        ceiling={b: float(D_LEAD[BEN.index(b)]) for b in PICK})
labels = [b["args"][1]["title.text"].split("— ")[1] for b in fig.layout.updatemenus[0].buttons]
# the repo figure draws the curve and the dots; the asymptotes it implies are
# added here so the ceiling is a labelled line and not a plateau to eyeball.
for b in labels:
    i = BEN.index(b)
    lo, hi = ETA[keep][np.array(bench_of) == b].min() - 1, ETA[keep][np.array(bench_of) == b].max() + 1
    fig.add_trace(go.Scatter(x=[lo, hi], y=[D_LEAD[i]] * 2, mode="lines",
                             name=f"ceiling d = {D_LEAD[i]:.3f}"
                                  + (f" (fixed {FIXED_D[i]:.2f}, shaved)" if FIXED_D[i] < 1 else ""),
                             line=dict(color="#0072B2", width=1.6, dash="dash"), hoverinfo="skip"))
    fig.add_trace(go.Scatter(x=[lo, hi], y=[FLOOR[i]] * 2, mode="lines",
                             name=f"chance floor c = {FLOOR[i]:.3f}",
                             line=dict(color="#999999", width=1.6, dash="dot"), hoverinfo="skip"))
nb_ = len(labels)
j = labels.index("MMLU")
for jj, btn in enumerate(fig.layout.updatemenus[0].buttons):
    vis = [False] * (4 * nb_)
    for t_ in (2 * jj, 2 * jj + 1, 2 * nb_ + 2 * jj, 2 * nb_ + 2 * jj + 1):
        vis[t_] = True
    btn["args"][0]["visible"] = vis
for t in fig.data:
    t.visible = False
for t_ in (2 * j, 2 * j + 1, 2 * nb_ + 2 * j, 2 * nb_ + 2 * j + 1):
    fig.data[t_].visible = True
fig.layout.updatemenus[0].active = j
fig.update_layout(title_text="Item characteristic curve — MMLU (dropdown: 8 benchmarks)",
                  height=560, width=1000, margin=dict(t=110, r=30))
show(fig, "01_icc")

print(f"{'benchmark':26s} {'n':>4s} {'c':>6s} {'d_fix':>6s} {'d_post':>7s} "
      f"{'max obs':>8s} {'headroom':>9s} {'(max-c)/(d-c)':>14s}")
for b in PICK:
    i = BEN.index(b)
    print(f"{b:26s} {NOBS_B[i]:4d} {FLOOR[i]:6.3f} {FIXED_D[i]:6.2f} {DMEAN[i]:7.3f} "
          f"{MAXOBS[i]:8.3f} {HEAD[i]:9.3f} {APPR[i]:14.3f}")

benchmark                     n      c  d_fix  d_post  max obs  headroom  (max-c)/(d-c)
MMLU                        138  0.250   1.00   0.894    0.898    -0.004          1.006
SWE-Bench Verified           33  0.000   1.00   0.839    0.835     0.004          0.995
WMDP Biology                 34  0.250   1.00   0.888    0.875     0.013          0.980
FrontierMath v1             101  0.000   0.57   0.545    0.524     0.021          0.961
FrontierMath Tier 4 v1       72  0.000   0.60   0.587    0.479     0.108          0.816
ARC-AGI-2                   154  0.000   1.00   0.996    0.999    -0.003          1.003
MATH Level 5                111  0.000   1.00   0.993    0.981     0.012          0.988
OTIS Mock AIME 2024-2025    158  0.001   1.00   0.993    0.999    -0.006          1.006


### 2 · The decisive test: a ceiling is data-identified only where scores press it. **6 of 98** gaps clear the prior 95th percentile, and every one of them has headroom **≤ 0.051**.
FrontierMath v1 tops out at **0.524** against its fixed d = **0.57** (headroom 0.046 before the estimated shave, 0.021 after); Tier 4 v1 at **0.479** against **0.60** (0.121). Neither wall could have been learned — the estimated gap only shaves d downward, and on Tier 4 v1 it is **0.015 ± 0.020**, the prior itself.

In [3]:
LAB = {"MMLU": (-50, 20), "SWE-Bench Verified": (78, -6), "WMDP Biology": (74, 16),
       "FrontierMath v1": (86, -4), "FrontierMath Tier 4 v1": (96, 18),
       "CritPt": (-4, -28), "ForecastBench": (-10, -30)}
SEL = ["SWE-Bench Verified", "WMDP Biology", "MMLU", "MMLU Pro Biology",
       "GPQA Diamond", "FrontierMath v1", "ARC-AGI-2", "OTIS Mock AIME 2024-2025",
       "MATH Level 5", "Aider Polyglot", "FrontierMath Tier 4 v1", "SimpleQA Verified",
       "Visual Task Assessment (VISTA)", "ForecastBench", "CritPt"]
assert set(SEL) <= set(BEN), set(SEL) - set(BEN)
si = [BEN.index(b) for b in SEL]
si = [si[i] for i in np.argsort([-HEAD[i] for i in si])]   # least headroom on top

fig = make_subplots(rows=1, cols=2, column_widths=[0.47, 0.53], horizontal_spacing=0.11,
                    subplot_titles=["All 98: gap posterior vs headroom left under d",
                                    f"Where the scores actually stop, {len(SEL)} benchmarks"])
for tag, m, col, sym, bars in [
        (f"gap past the prior 95th pct ({int((OFF & (FIXED_D == 1)).sum())}), ± 1 SD",
         OFF & (FIXED_D == 1), C["verm"], "circle", True),
        (f"gap at the prior ({int((~OFF & (FIXED_D == 1)).sum())}), SD bars omitted",
         ~OFF & (FIXED_D == 1), C["gray"], "circle", False),
        ("fixed d < 1 from ground truth (2), ± 1 SD", FIXED_D < 1, C["blue"], "star", True)]:
    idx = np.flatnonzero(m)
    fig.add_trace(go.Scatter(
        x=HEAD[idx], y=GMED[idx], mode="markers", name=tag,
        marker=dict(color=col, symbol=sym, size=6 + 9 * NOBS_B[idx] / NOBS_B.max(),
                    opacity=0.85, line=dict(color="white", width=0.7)),
        error_y=dict(type="data", array=GSD[idx] if bars else np.zeros(idx.size), color=col,
                     thickness=1.0, width=0),
        text=[BEN[i] for i in idx],
        hovertemplate="%{text}<br>headroom %{x:.3f}<br>gap %{y:.3f}<extra></extra>"),
        row=1, col=1)
fig.add_trace(go.Scatter(x=[-0.05, 0.70], y=[P_Q95] * 2, mode="lines",
                         name=f"prior 95th pct {P_Q95:.3f}",
                         line=dict(color=C["dark"], width=1.5, dash="dash")), row=1, col=1)
fig.add_trace(go.Scatter(x=[-0.05, 0.70], y=[P_MED] * 2, mode="lines",
                         name=f"prior median {P_MED:.3f}",
                         line=dict(color=C["dark"], width=1.5, dash="dot")), row=1, col=1)
for b, (ax_, ay_) in LAB.items():
    i = BEN.index(b)
    fig.add_annotation(x=HEAD[i], y=GMED[i], text=b, showarrow=True, arrowhead=2,
                       ax=ax_, ay=ay_, font=dict(size=9.5, color=C["dark"]),
                       arrowcolor=C["dark"], arrowwidth=0.8, row=1, col=1)
fig.update_xaxes(title_text="headroom  d − max observed score", range=[-0.05, 0.70], row=1, col=1)
fig.update_yaxes(title_text="ceiling_gap posterior median", range=[-0.012, 0.235], row=1, col=1)

names = [BEN[i] for i in si]
for i in si:
    fig.add_trace(go.Scatter(x=[FLOOR[i], DMEAN[i]], y=[BEN[i]] * 2, mode="lines",
                             line=dict(color=C["gray"], width=5), showlegend=False,
                             hoverinfo="skip"), row=1, col=2)
fig.add_trace(go.Scatter(x=[FLOOR[i] for i in si], y=names, mode="markers",
                         name="chance floor c (bar spans c to d)",
                         marker=dict(color=C["dark"], size=7, symbol="line-ns-open",
                                     line=dict(color=C["dark"], width=2)),
                         hovertemplate="%{y}: c = %{x:.3f}<extra></extra>"), row=1, col=2)
fig.add_trace(go.Scatter(x=[MAXOBS[i] for i in si], y=names, mode="markers",
                         name="highest observed score",
                         marker=dict(color=C["orange"], size=10, symbol="diamond",
                                     line=dict(color="white", width=1)),
                         hovertemplate="%{y}: max obs %{x:.3f}<extra></extra>"), row=1, col=2)
fig.add_trace(go.Scatter(
    x=[DMEAN[i] for i in si], y=names, mode="markers", name="posterior-mean ceiling d",
    marker=dict(color=C["blue"], size=9, line=dict(color="white", width=1)),
    error_x=dict(type="data", symmetric=False,
                 array=[DMEAN[i] - (FIXED_D[i] - GQ[0][i] * (FIXED_D[i] - FLOOR[i])) for i in si],
                 arrayminus=[(FIXED_D[i] - GQ[1][i] * (FIXED_D[i] - FLOOR[i])) - DMEAN[i] for i in si],
                 color=C["blue"], thickness=1.3, width=4),
    hovertemplate="%{y}: d = %{x:.3f}<extra></extra>"), row=1, col=2)
fig.add_trace(go.Scatter(x=[FIXED_D[i] for i in si if FIXED_D[i] < 1],
                         y=[BEN[i] for i in si if FIXED_D[i] < 1], mode="markers",
                         name="fixed d before the estimated shave",
                         marker=dict(color=C["blue"], size=13, symbol="star-open",
                                     line=dict(color=C["blue"], width=1.4)),
                         hovertemplate="%{y}: fixed d = %{x:.2f}<extra></extra>"), row=1, col=2)
fig.update_xaxes(title_text="score scale", range=[-0.03, 1.05], row=1, col=2)
fig.update_yaxes(categoryorder="array", categoryarray=names, tickfont=dict(size=9.5), row=1, col=2)
fig.update_layout(title="Inferred where the data press the wall, prior-driven everywhere else",
                  height=600, width=1280, legend=dict(orientation="h", y=-0.20),
                  margin=dict(t=100, b=140, l=40, r=20))
show(fig, "02_headroom")

print(f"gaps past the prior 95th pct ({P_Q95:.3f}): {int(OFF.sum())} of {NB} · "
      f"largest headroom among them {HEAD[OFF].max():.3f}")
for b in sorted([BEN[i] for i in np.flatnonzero(OFF)], key=lambda b: -GMED[BEN.index(b)]):
    i = BEN.index(b)
    print(f"  {b:24s} gap {GMED[i]:.3f} ± {GSD[i]:.3f} · d {DMEAN[i]:.3f} · "
          f"max obs {MAXOBS[i]:.3f} · headroom {HEAD[i]:+.3f} · n {NOBS_B[i]}")
for b in sorted(CAPPED):
    i = BEN.index(b)
    print(f"  {b:24s} fixed d {FIXED_D[i]:.2f} · max obs {MAXOBS[i]:.3f} → headroom "
          f"{FIXED_D[i] - MAXOBS[i]:+.3f} BEFORE the shave, {HEAD[i]:+.3f} after; "
          f"gap {GMED[i]:.3f} ± {GSD[i]:.3f} vs prior {P_MED:.3f} ± {P_SD:.3f}")
print("no observation anywhere in the data exceeds 0.524 / 0.479 on those two, so d = 0.57 / "
      "0.60 is not a quantity this fit could have estimated: it is carried in from Epoch's "
      "published score ceilings and the noise gap can only move it down.")

gaps past the prior 95th pct (0.059): 6 of 98 · largest headroom among them 0.051
  SWE-Bench Verified       gap 0.175 ± 0.046 · d 0.839 · max obs 0.835 · headroom +0.004 · n 33
  WMDP Biology             gap 0.159 ± 0.034 · d 0.888 · max obs 0.875 · headroom +0.013 · n 34
  MMLU                     gap 0.140 ± 0.039 · d 0.894 · max obs 0.898 · headroom -0.004 · n 138
  MMLU Pro Biology         gap 0.072 ± 0.026 · d 0.941 · max obs 0.922 · headroom +0.019 · n 33
  BoolQ                    gap 0.067 ± 0.051 · d 0.963 · max obs 0.912 · headroom +0.051 · n 78
  BIG-Bench Hard (BBH)     gap 0.061 ± 0.032 · d 0.951 · max obs 0.944 · headroom +0.007 · n 52
  FrontierMath Tier 4 v1   fixed d 0.60 · max obs 0.479 → headroom +0.121 BEFORE the shave, +0.108 after; gap 0.015 ± 0.020 vs prior 0.014 ± 0.020
  FrontierMath v1          fixed d 0.57 · max obs 0.524 → headroom +0.046 BEFORE the shave, +0.021 after; gap 0.040 ± 0.029 vs prior 0.014 ± 0.020
no observation anywhere in the data exceeds 0.5

### 3 · Nearly every gap posterior is its prior: median **0.015** against the prior's **0.014**, and 92 of 98 stay inside the prior's 95% mass.
Only 6 move: the 3 movers above plus MMLU Pro Biology, BoolQ and BIG-Bench Hard (BBH). Posterior SD falls below the prior's **0.020** on **23** benchmarks, so the data speak a little more widely than they move the location.

In [4]:
grid = np.linspace(0, 0.26, 400)
o = np.argsort(-GMED)[:14][::-1]
fig = make_subplots(rows=1, cols=2, column_widths=[0.52, 0.48], horizontal_spacing=0.11,
                    subplot_titles=["98 gap posteriors against the Beta(1, 49) prior",
                                    "The 14 largest, with 5–95% posterior interval"])
fig.add_trace(go.Scatter(x=grid, y=PRI.pdf(grid), mode="lines", name="prior Beta(1, 49) density",
                         line=dict(color=C["dark"], width=1.8), fill="tozeroy",
                         fillcolor="rgba(51,51,51,0.10)", hoverinfo="skip"), row=1, col=1)
for tag, m, col in [("posterior median, past the prior 95th pct (6)", OFF, C["verm"]),
                    ("posterior median, inside it (92)", ~OFF, C["gray"])]:
    idx = np.flatnonzero(m)
    fig.add_trace(go.Scatter(x=GMED[idx], y=np.full(idx.size, -2.4), mode="markers", name=tag,
                             marker=dict(color=col, size=13, symbol="line-ns-open",
                                         line=dict(color=col, width=1.6)),
                             text=[BEN[i] for i in idx],
                             hovertemplate="%{text}: %{x:.3f}<extra></extra>"), row=1, col=1)
fig.add_trace(go.Scatter(x=[P_Q95] * 2, y=[-4, 50], mode="lines",
                         name=f"prior 95th pct {P_Q95:.3f}",
                         line=dict(color=C["verm"], width=1.5, dash="dash")), row=1, col=1)
fig.update_xaxes(title_text="ceiling_gap", range=[-0.004, 0.22], row=1, col=1)
fig.update_yaxes(title_text="prior density (rug = one benchmark)", range=[-5, 50], row=1, col=1)

fig.add_trace(go.Scatter(
    x=GMED[o], y=[BEN[i] for i in o], mode="markers", name="posterior median (5–95%)",
    marker=dict(color=[C["verm"] if OFF[i] else C["gray"] for i in o], size=10,
                line=dict(color="white", width=1)),
    error_x=dict(type="data", symmetric=False, array=GQ[1][o] - GMED[o],
                 arrayminus=GMED[o] - GQ[0][o], color=C["dark"], thickness=1.2, width=4),
    hovertemplate="%{y}: %{x:.3f}<extra></extra>"), row=1, col=2)
fig.add_trace(go.Scatter(x=[P_MED] * 2, y=[BEN[o[0]], BEN[o[-1]]], mode="lines",
                         name=f"prior median {P_MED:.3f}",
                         line=dict(color=C["dark"], width=1.5, dash="dot"), hoverinfo="skip"),
              row=1, col=2)
fig.add_trace(go.Scatter(x=[P_Q95] * 2, y=[BEN[o[0]], BEN[o[-1]]], mode="lines", showlegend=False,
                         line=dict(color=C["verm"], width=1.5, dash="dash"), hoverinfo="skip"),
              row=1, col=2)
fig.update_xaxes(title_text="ceiling_gap", range=[-0.005, 0.27], row=1, col=2)
fig.update_yaxes(categoryorder="array", categoryarray=[BEN[i] for i in o],
                 tickfont=dict(size=9.5), row=1, col=2)
fig.update_layout(title="The estimated noise gap moves on 6 of 98 benchmarks",
                  height=560, width=1250, legend=dict(orientation="h", y=-0.17),
                  margin=dict(t=100, b=120, l=40, r=30))
show(fig, "03_gap_vs_prior")

print(f"prior Beta(1,49): mean {1 / 50:.4f} median {P_MED:.4f} sd {P_SD:.4f} "
      f"95th pct {P_Q95:.4f}")
print(f"posterior medians: median across benchmarks {np.median(GMED):.4f} · "
      f"{int(OFF.sum())} past the prior 95th pct · {int((GSD < P_SD).sum())} with SD below "
      f"the prior SD · {int((GMED > 0.10).sum())} past 0.10")
print("moved: " + " · ".join(f"{BEN[i]} {GMED[i]:.3f}" for i in np.argsort(-GMED)[:6]))

prior Beta(1,49): mean 0.0200 median 0.0140 sd 0.0196 95th pct 0.0593
posterior medians: median across benchmarks 0.0153 · 6 past the prior 95th pct · 23 with SD below the prior SD · 3 past 0.10
moved: SWE-Bench Verified 0.175 · WMDP Biology 0.159 · MMLU 0.140 · MMLU Pro Biology 0.072 · BoolQ 0.067 · BIG-Bench Hard (BBH) 0.061


### 4 · Six chains, **three** basins at alignment 0.95: {0,2,3} at logp **3444.2**, chain 5 alone at **3441.7**, {1,4} at **3430.6**.
Chain 5 matches the leading three at **0.91** and the other two at **0.56–0.57**: a near-neighbour of the leader, not a member. Lower the threshold to 0.90 and it merges in — which is exactly why the merged group reads as unconverged in the next claim.

In [5]:
ORD = LEAD + CH5 + ALT
tick = [f"c{c}" for c in ORD]
fig = make_subplots(rows=1, cols=2, column_widths=[0.44, 0.56], horizontal_spacing=0.13,
                    subplot_titles=["Log posterior density per chain",
                                    "Chain-pair alignment (worst matched axis)"])
for g in G95:
    t, col = tuple(g), BCOL[tuple(g)]
    fig.add_trace(go.Scatter(
        x=[f"c{c}" for c in g], y=LP[g], mode="markers", name=BNAME[t],
        marker=dict(color=col, size=12, symbol="diamond", line=dict(color="white", width=1)),
        error_y=dict(type="data", symmetric=False, array=LPQ[1][g] - LP[g],
                     arrayminus=LP[g] - LPQ[0][g], color=col, thickness=1.3, width=5),
        hovertemplate="%{x}: %{y:.1f}<extra></extra>"), row=1, col=1)
    fig.add_trace(go.Scatter(x=tick, y=[LP[g].mean()] * len(tick), mode="lines",
                             name=f"{BNAME[t].split(' · ')[0]} mean logp {LP[g].mean():.1f}",
                             line=dict(color=col, width=1, dash="dot"), hoverinfo="skip"),
                  row=1, col=1)
fig.add_annotation(x="c5", y=(LP[LEAD].mean() + LP[ALT].mean()) / 2,
                   text=f"<b>{LP[LEAD].mean() - LP[ALT].mean():.1f} nats</b>", showarrow=True,
                   arrowhead=2, ax=40, ay=0, font=dict(size=11.5, color=C["dark"]),
                   arrowcolor=C["dark"], row=1, col=1)
Z = ALIGN[np.ix_(ORD, ORD)]
fig.add_trace(go.Heatmap(
    z=Z, x=tick, y=tick, zmin=0.5, zmax=1.0, texttemplate="%{z:.2f}", textfont=dict(size=11), colorscale=[[0, "#FFF7EC"], [0.5, C["orange"]], [1.0, C["blue"]]],
    colorbar=dict(title="worst matched<br>axis corr", thickness=12, len=0.80, x=1.005, y=0.44),
    hovertemplate="%{y} vs %{x}: %{z:.3f}<extra></extra>"), row=1, col=2)
for pos in (len(LEAD) - 0.5, len(LEAD) + len(CH5) - 0.5):
    fig.add_vline(x=pos, line=dict(color=C["dark"], width=2), row=1, col=2)
    fig.add_hline(y=pos, line=dict(color=C["dark"], width=2), row=1, col=2)
fig.update_xaxes(categoryorder="array", categoryarray=tick, title_text="chain", row=1, col=1)
fig.update_xaxes(title_text="chain, grouped A | B | C (rules = 0.95 boundaries)", row=1, col=2)
fig.update_yaxes(title_text="mean logp (5–95% of draws)", row=1, col=1)
fig.update_yaxes(autorange="reversed", row=1, col=2)
fig.update_layout(title="Three basins at 0.95; chain 5 is a 0.91 neighbour of the leader",
                  height=500, width=1200, legend=dict(orientation="h", y=-0.22),
                  margin=dict(t=90, b=120, r=110))
show(fig, "04_basin_census")

for g in G95:
    print(f"{BNAME[tuple(g)]:24s} logp {LP[g].mean():7.1f} · divergences "
          f"{int(DIV[g].sum()):3d} · within-basin worst align "
          f"{ALIGN[np.ix_(g, g)].min():.3f}")
print(f"chain 5 vs leading {ALIGN[5, LEAD].min():.2f}–{ALIGN[5, LEAD].max():.2f} · "
      f"vs {{1,4}} {ALIGN[5, ALT].min():.2f}–{ALIGN[5, ALT].max():.2f} → its own basin at "
      f"0.95, folded into the leader at 0.90")
print(f"clustering at 0.90 gives {G90}, at 0.95 gives {G95}")

basin A · chains 0,2,3   logp  3444.2 · divergences   0 · within-basin worst align 0.999
basin B · chain 5        logp  3441.7 · divergences   0 · within-basin worst align 1.000
basin C · chains 1,4     logp  3430.6 · divergences  18 · within-basin worst align 0.999
chain 5 vs leading 0.91–0.91 · vs {1,4} 0.56–0.57 → its own basin at 0.95, folded into the leader at 0.90
clustering at 0.90 gives [[0, 2, 3, 5], [1, 4]], at 0.95 gives [[0, 2, 3], [5], [1, 4]]


### 5 · Only the 0.95 leading basin converges: bulk ESS(D) min/median **333 / 1498** and eta r̂ **1.025**, against **10 / 37** and **1.549** pooled.
The 0.90 merge is the trap: adding chain 5 drops median ESS to **408** and lifts eta r̂ to **1.473**. All **18 / 12,000** divergences sit in chain 1. Fit is indifferent — every basin lands within 0.001 RMSE of the others, and each basin's R² (**0.9691–0.9703**) clears the stored pooled Bayesian R² of **0.9574**, which is computed over draws and so pays for averaging two solutions.

In [6]:
NMS = list(SETS)
NCOL = {NMS[0]: C["dark"], NMS[1]: C["orange"], NMS[2]: C["blue"], NMS[3]: C["verm"]}
fig = make_subplots(rows=1, cols=3, column_widths=[0.34, 0.33, 0.33], horizontal_spacing=0.10,
                    subplot_titles=["Bulk ESS on D (98 benchmarks)",
                                    "Identified r̂ (permutation-invariant)",
                                    "Fit on all 4,445 observations"])
for nm in NMS:
    col, sel = NCOL[nm], SETS[nm]
    fig.add_trace(go.Bar(x=["ESS min", "ESS median"], y=[ESS[nm][0], ESS[nm][1]],
                         name=f"{nm} · {int(DIV[sel].sum())} div / {2000 * len(sel):,}",
                         legendgroup=nm, marker_color=col,
                         text=[f"{ESS[nm][0]:.0f}", f"{ESS[nm][1]:.0f}"],
                         textposition="outside", textfont=dict(size=9.5),
                         hovertemplate="%{x}: %{y:.0f}<extra></extra>"), row=1, col=1)
    r = RH[nm]
    fig.add_trace(go.Bar(x=["eta", "D", "sigma_b"],
                         y=[r["eta_max_rhat"], r["D_max_rhat"], r["sigma_b_max_rhat"]],
                         legendgroup=nm, showlegend=False, marker_color=col,
                         text=[f"{r['eta_max_rhat']:.3f}", f"{r['D_max_rhat']:.3f}",
                               f"{r['sigma_b_max_rhat']:.3f}"],
                         textposition="outside", textfont=dict(size=9),
                         hovertemplate="%{x}: %{y:.3f}<extra></extra>"), row=1, col=2)
    g_ = GF[nm]
    fig.add_trace(go.Bar(x=["RMSE", "MAE", "1 − R²"], y=[g_["rmse"], g_["mae"], 1 - g_["r2"]],
                         legendgroup=nm, showlegend=False, marker_color=col,
                         text=[f"{g_['rmse']:.4f}", f"{g_['mae']:.4f}", f"R² {g_['r2']:.4f}"],
                         textposition="inside", insidetextanchor="start", textangle=-90,
                         textfont=dict(size=8.5, color="white"),
                         hovertemplate="%{x}: %{y:.4f}<extra></extra>"), row=1, col=3)
fig.add_trace(go.Scatter(x=["eta", "D", "sigma_b"], y=[1.01] * 3, mode="lines",
                         name="r̂ = 1.01 (convergence bar)",
                         line=dict(color=C["dark"], width=1.5, dash="dot")), row=1, col=2)
fig.add_trace(go.Scatter(x=["RMSE", "MAE", "1 − R²"], y=[1 - GOF_FILE["bayesian_r2"]] * 3,
                         mode="lines",
                         name=f"1 − R² of the stored pooled GoF ({GOF_FILE['bayesian_r2']:.4f})",
                         line=dict(color=C["green"], width=1.5, dash="dash")), row=1, col=3)
fig.update_yaxes(title_text="effective draws", type="log", range=[0.75, 3.42],
                 tickmode="array", tickvals=[10, 100, 1000], ticktext=["10", "100", "1000"],
                 row=1, col=1)
fig.update_yaxes(title_text="r̂", range=[1.0, 1.80], row=1, col=2)
fig.update_yaxes(title_text="error (third group: 1 − R², labelled with R²)",
                 range=[0, 0.062], row=1, col=3)
fig.update_layout(title="Convergence and fit, pooled and per basin", height=520, width=1320,
                  bargap=0.26, legend=dict(orientation="h", y=-0.20),
                  margin=dict(t=100, b=130, l=70, r=30))
show(fig, "05_convergence")

for nm in NMS:
    r, g_ = RH[nm], GF[nm]
    print(f"{nm:24s} ESS(D) min {ESS[nm][0]:6.0f} med {ESS[nm][1]:7.0f} · div "
          f"{int(DIV[SETS[nm]].sum()):3d}/{2000 * len(SETS[nm]):6d} · eta r̂ "
          f"{r['eta_max_rhat']:.3f} D {r['D_max_rhat']:.3f} sigma_b "
          f"{r['sigma_b_max_rhat']:.3f} · RMSE {g_['rmse']:.4f} MAE {g_['mae']:.4f} "
          f"R² {g_['r2']:.4f}")
print(f"basin B · chain 5 alone: RMSE {GF[BNAME[tuple(CH5)]]['rmse']:.4f} MAE "
      f"{GF[BNAME[tuple(CH5)]]['mae']:.4f} R² {GF[BNAME[tuple(CH5)]]['r2']:.4f} "
      f"(one chain, so no r̂ and no cross-chain ESS)")
print(f"stored pooled GoF on file: RMSE {GOF_FILE['rmse']:.6f} MAE {GOF_FILE['mae']:.6f} "
      f"R² {GOF_FILE['bayesian_r2']:.6f} — below every within-basin R² because the pooled "
      f"posterior mean averages two different solutions")

pooled (6 chains)        ESS(D) min     10 med      37 · div  18/ 12000 · eta r̂ 1.549 D 1.631 sigma_b 1.497 · RMSE 0.0458 MAE 0.0306 R² 0.9708
0.90 merge · 0,2,3,5     ESS(D) min      9 med     408 · div   0/  8000 · eta r̂ 1.473 D 1.369 sigma_b 1.414 · RMSE 0.0461 MAE 0.0307 R² 0.9704
basin A · chains 0,2,3   ESS(D) min    333 med    1498 · div   0/  6000 · eta r̂ 1.025 D 1.017 sigma_b 1.013 · RMSE 0.0462 MAE 0.0308 R² 0.9703
basin C · chains 1,4     ESS(D) min     41 med     990 · div  18/  4000 · eta r̂ 1.056 D 1.048 sigma_b 1.045 · RMSE 0.0467 MAE 0.0311 R² 0.9696
basin B · chain 5 alone: RMSE 0.0471 MAE 0.0311 R² 0.9691 (one chain, so no r̂ and no cross-chain ESS)
stored pooled GoF on file: RMSE 0.045823 MAE 0.030620 R² 0.957421 — below every within-basin R² because the pooled posterior mean averages two different solutions


### 6 · What changes on axis 2 is GBAEval: loading **2.41 / 2.26** in basins A and B, **0.19** in basin C, where it reappears on the fluid axis at **2.20**.
The 7 classic commonsense items keep the axis in all three basins (mean loading **1.41 / 1.31 / 1.36**) and hold their ranking. The anomaly is that GBAEval outranks every one of them in basins A and B while being categorised Agentic Computer Use with **zero** shared models against all 7.

In [7]:
MARK = set(CLASSIC) | {"GBAEval"}
sel = sorted({i for g in G95 for i in np.argsort(-AB[tuple(g)][:, K2])[:12]} | set(CI) | {GBI},
             key=lambda i: AB[tuple(LEAD)][i, K2])
def ylab(i):
    c = str(CAT.get(BEN[i], "?"))
    nm = f"<b>{BEN[i]}</b>" if BEN[i] in MARK else BEN[i]
    return f"{nm} <span style='font-size:9px;color:#777'>· {c}</span>"
Y = [ylab(i) for i in sel]

fig = make_subplots(rows=1, cols=2, column_widths=[0.62, 0.38], horizontal_spacing=0.05,
                    shared_yaxes=False,
                    subplot_titles=["Each basin's top 12, plus all 7 classic items",
                                    "Where GBAEval goes instead"])
for g in G95:
    t = tuple(g)
    fig.add_trace(go.Bar(x=AB[t][sel, K2], y=Y, orientation="h", name=BNAME[t],
                         legendgroup=BNAME[t], marker_color=BCOL[t],
                         hovertemplate="%{y}: %{x:.2f}<extra></extra>"), row=1, col=1)
fig.add_trace(go.Bar(x=[None], y=[None], marker_color="rgba(0,0,0,0)",
                     name="bold = one of the 7 classic items, or GBAEval"), row=1, col=1)
for g in G95:
    t = tuple(g)
    fig.add_trace(go.Bar(x=AXIS, y=AB[t][GBI], name=BNAME[t], legendgroup=BNAME[t],
                         showlegend=False, marker_color=BCOL[t],
                         text=[f"{v:.2f}" for v in AB[t][GBI]], textposition="outside",
                         textfont=dict(size=9.5),
                         hovertemplate="%{x}: %{y:.2f}<extra></extra>"), row=1, col=2)
fig.add_trace(go.Scatter(x=AXIS, y=[AB[tuple(LEAD)][CI, K2].mean()] * 3, mode="lines",
                         name=f"mean of the 7 classic items on axis 2, basin A "
                              f"({AB[tuple(LEAD)][CI, K2].mean():.2f})",
                         line=dict(color=C["dark"], width=1.5, dash="dot")), row=1, col=2)
fig.update_xaxes(title_text=f"loading on axis 2 ({AXIS[K2]})", range=[0, 2.62], row=1, col=1)
fig.update_yaxes(categoryorder="array", categoryarray=Y, tickfont=dict(size=9.5), row=1, col=1)
fig.update_xaxes(title_text="axis", tickfont=dict(size=9), row=1, col=2)
fig.update_yaxes(title_text="GBAEval loading", range=[0, 2.75], row=1, col=2)
fig.update_layout(title="Axis 2 across the three basins: the classic core holds, GBAEval moves",
                  height=680, width=1300, barmode="group", bargap=0.22,
                  legend=dict(orientation="h", y=-0.13), margin=dict(t=100, b=110, l=280))
show(fig, "06_axis2_loadings")

for g in G95:
    t = tuple(g)
    print(f"{BNAME[t]:24s} classic mean per axis {np.round(AB[t][CI].mean(axis=0), 2)} · "
          f"GBAEval {np.round(AB[t][GBI], 2)} (axes {AXIS})")
best = max(CI, key=lambda i: AB[tuple(LEAD)][i, K2])
print(f"strongest classic item on axis 2, basin A: {BEN[best]} "
      f"{AB[tuple(LEAD)][best, K2]:.2f}; GBAEval is "
      f"{100 * (AB[tuple(LEAD)][GBI, K2] / AB[tuple(LEAD)][best, K2] - 1):+.0f}% above it, "
      f"categorised {str(CAT.get('GBAEval'))!r}")
RAWB = pd.read_csv(ROOT / "1_data/processed/benchmarks_merged.csv")
mods = {b: set(RAWB.loc[RAWB["benchmark"] == b, "model_version"]) for b in MARK}
print("GBAEval shared models with each classic item: " +
      " · ".join(f"{c} {len(mods['GBAEval'] & mods[c])}" for c in CLASSIC))

basin A · chains 0,2,3   classic mean per axis [0.28 1.41 0.3 ] · GBAEval [0.5  2.41 0.28] (axes ['Hard math + science', 'Easy knowledge / commonsense', 'Fluid / abstract'])
basin B · chain 5        classic mean per axis [0.18 1.31 0.44] · GBAEval [0.72 2.26 0.25] (axes ['Hard math + science', 'Easy knowledge / commonsense', 'Fluid / abstract'])
basin C · chains 1,4     classic mean per axis [0.28 1.36 0.26] · GBAEval [0.24 0.19 2.2 ] (axes ['Hard math + science', 'Easy knowledge / commonsense', 'Fluid / abstract'])
strongest classic item on axis 2, basin A: GSM8K 2.11; GBAEval is +14% above it, categorised 'Agentic Computer Use'
GBAEval shared models with each classic item: GSM8K 0 · OpenBookQA 0 · ARC (AI2) 0 · PIQA 0 · HellaSwag 0 · Adversarial NLI 0 · CSQA2 0


### 7 · On the benchmark side the split is confined to axis 2: loadings correlate **0.91** on hard math and **0.76** on fluid across the two largest groups, but only **0.60** on axis 2.
Basins compared as the 0.90 grouping, {0,2,3,5} against {1,4}. The axis-2 scatter is a single outlier story — drop GBAEval and it rises to **0.68**.

In [8]:
A0, A1 = AB90[tuple(G90[0])], AB90[tuple(G90[1])]
keep = np.arange(NB) != GBI
lim = 1.06 * max(A0.max(), A1.max())
fig = go.Figure()
fig.add_trace(go.Scatter(x=[0, lim], y=[0, lim], mode="lines", name="identity",
                         line=dict(color=C["dark"], width=1.2, dash="dot"), hoverinfo="skip"))
for k in range(3):
    r_ = np.corrcoef(A0[:, k], A1[:, k])[0, 1]
    rk = np.corrcoef(A0[keep, k], A1[keep, k])[0, 1]
    fig.add_trace(go.Scatter(
        x=A0[:, k], y=A1[:, k], mode="markers",
        name=f"{AXIS[k]} · r = {r_:.2f} (without GBAEval {rk:.2f})",
        marker=dict(color=AXCOL[k], size=8, opacity=0.8, line=dict(color="white", width=0.7)),
        text=BEN, hovertemplate="%{text}<br>{0,2,3,5} %{x:.2f} → {1,4} %{y:.2f}<extra></extra>"))
AOFF = {0: (96, 24), 1: (16, -30), 2: (92, -6)}          # per-axis label placement
for k in range(3):
    fig.add_annotation(x=A0[GBI, k], y=A1[GBI, k], text=f"GBAEval · {AXIS[k].split(' /')[0]}",
                       showarrow=True, arrowhead=2, ax=AOFF[k][0], ay=AOFF[k][1],
                       font=dict(size=9.5, color=C["dark"]), arrowcolor=C["dark"],
                       arrowwidth=0.8)
fig.update_layout(
    title="Per-benchmark loadings across the two largest groups, one point per benchmark per axis",
    xaxis=dict(title="loading in {0,2,3,5} (the 0.90 leading group)", range=[0, lim],
               constrain="domain"),
    yaxis=dict(title="loading in {1,4}", range=[0, lim], scaleanchor="x", scaleratio=1,
               constrain="domain"),
    height=680, width=760, legend=dict(orientation="h", y=-0.14), margin=dict(t=90, b=130))
show(fig, "07_axes13_loadings")

for k in range(3):
    print(f"{AXIS[k]:30s} loading pearson {np.corrcoef(A0[:, k], A1[:, k])[0, 1]:+.3f} "
          f"spearman {spearmanr(A0[:, k], A1[:, k]).statistic:+.3f} · without GBAEval "
          f"{np.corrcoef(A0[keep, k], A1[keep, k])[0, 1]:+.3f}")
print(f"GBAEval: {np.round(A0[GBI], 2)} → {np.round(A1[GBI], 2)} on axes {AXIS}")

Hard math + science            loading pearson +0.909 spearman +0.875 · without GBAEval +0.910
Easy knowledge / commonsense   loading pearson +0.596 spearman +0.427 · without GBAEval +0.677
Fluid / abstract               loading pearson +0.764 spearman +0.787 · without GBAEval +0.805
GBAEval: [0.56 2.37 0.27] → [0.24 0.19 2.2 ] on axes ['Hard math + science', 'Easy knowledge / commonsense', 'Fluid / abstract']


### 8 · The ranking that the index reports is mode-stable: Spearman **+0.983** on the summed axes and **+0.967** on hard math, against **+0.700** on axis 2 and **+0.844** on fluid.
Same 0.90 grouping, all 765 test-takers. Axis 2 is the only axis whose per-model ordering the basins disagree on, and the disagreement cancels in the sum.

In [9]:
T0, T1 = TB90[tuple(G90[0])], TB90[tuple(G90[1])]
rho = [spearmanr(T0[:, k], T1[:, k]).statistic for k in range(3)]
rho_sum = spearmanr(T0.sum(1), T1.sum(1)).statistic
rho95 = [spearmanr(TB[tuple(LEAD)][:, k], TB[tuple(ALT)][:, k]).statistic for k in range(3)]
rho95_sum = spearmanr(TB[tuple(LEAD)].sum(1), TB[tuple(ALT)].sum(1)).statistic

fig = make_subplots(rows=1, cols=2, column_widths=[0.40, 0.60], horizontal_spacing=0.11,
                    subplot_titles=["Spearman of the model ordering across basins",
                                    "Rank against rank: the sum, and axis 2"])
xs = [AXIS[k].replace(" / ", "<br>").replace(" + ", "<br>+ ") for k in range(3)] + ["sum of<br>axes"]
fig.add_trace(go.Bar(x=xs, y=rho + [rho_sum], name="{0,2,3,5} vs {1,4} (0.90 grouping)",
                     marker_color=AXCOL + [C["dark"]],
                     text=[f"{v:+.3f}" for v in rho + [rho_sum]], textposition="inside",
                     insidetextanchor="start", textfont=dict(size=10.5, color="white"),
                     hovertemplate="%{x}: %{y:+.3f}<extra></extra>"),
              row=1, col=1)
fig.add_trace(go.Scatter(x=xs, y=rho95 + [rho95_sum], mode="markers",
                         name="{0,2,3} vs {1,4} (0.95 grouping)",
                         marker=dict(color=C["dark"], size=11, symbol="diamond-open",
                                     line=dict(color=C["dark"], width=1.6)),
                         hovertemplate="%{x}: %{y:+.3f}<extra></extra>"), row=1, col=1)
fig.update_yaxes(title_text="Spearman rho", range=[0, 1.13], row=1, col=1)
fig.update_xaxes(tickfont=dict(size=9), row=1, col=1)

rk = lambda v: np.argsort(np.argsort(v)) + 1
for tag, v0, v1, col in [(f"sum of axes · rho {rho_sum:+.3f}", T0.sum(1), T1.sum(1), C["dark"]),
                         (f"axis 2 ({AXIS[K2]}) · rho {rho[K2]:+.3f}",
                          T0[:, K2], T1[:, K2], AXCOL[K2])]:
    fig.add_trace(go.Scatter(x=rk(v0), y=rk(v1), mode="markers", name=tag,
                             marker=dict(color=col, size=4.5, opacity=0.55),
                             text=MOD, hovertemplate="%{text}<br>%{x} → %{y}<extra></extra>"),
                  row=1, col=2)
fig.add_trace(go.Scatter(x=[1, NM], y=[1, NM], mode="lines", name="identical ranking",
                         line=dict(color=C["gray"], width=1.4, dash="dot"), hoverinfo="skip"),
              row=1, col=2)
fig.update_xaxes(title_text="rank in {0,2,3,5}", row=1, col=2)
fig.update_yaxes(title_text="rank in {1,4}", row=1, col=2)
fig.update_layout(title="Model ordering survives the split except on axis 2",
                  height=520, width=1220, bargap=0.32,
                  legend=dict(orientation="h", y=-0.19), margin=dict(t=90, b=140))
show(fig, "08_rank_stability")

print("0.90 grouping {0,2,3,5} vs {1,4}: " +
      " · ".join(f"{AXIS[k]} {rho[k]:+.3f}" for k in range(3)) + f" · sum {rho_sum:+.3f}")
print("0.95 grouping {0,2,3}  vs {1,4}: " +
      " · ".join(f"{AXIS[k]} {rho95[k]:+.3f}" for k in range(3)) + f" · sum {rho95_sum:+.3f}")

0.90 grouping {0,2,3,5} vs {1,4}: Hard math + science +0.967 · Easy knowledge / commonsense +0.700 · Fluid / abstract +0.844 · sum +0.983
0.95 grouping {0,2,3}  vs {1,4}: Hard math + science +0.974 · Easy knowledge / commonsense +0.682 · Fluid / abstract +0.868 · sum +0.982


### 9 · LOO cannot separate the two largest basins: elpd difference **5.3 ± 22.3** nats, and the lower-logp basin is the one nominally ahead.
Both score the same 4,445 observations. **693** and **677** points sit at Pareto k > 0.7 with p_loo ≈ **1,460** on 4,445, so neither the magnitude nor the sign of that difference is readable. This contradicts the expectation that the ranking would at least be ordinal.

In [10]:
del post
ll = xr.open_dataset(TRACE, group="log_likelihood").isel(draw=slice(None, None, THIN)).load()
tiny = xr.open_dataset(TRACE, group="posterior")[["tau_A"]] \
         .isel(draw=slice(None, None, THIN)).load()
assert ll.sizes["obs_dim_0"] == len(data.scores) == 4445
loos = {}
for g in (LEAD, ALT):
    nm = BNAME[tuple(g)]
    loos[nm] = az.loo(az.InferenceData(posterior=tiny.isel(chain=g),
                                       log_likelihood=ll.isel(chain=g)), pointwise=True)
del ll, tiny
CMP = az.compare(loos, ic="loo")

order = [BNAME[tuple(ALT)], BNAME[tuple(LEAD)]]
fig = make_subplots(rows=1, cols=2, column_widths=[0.55, 0.45], horizontal_spacing=0.13,
                    subplot_titles=["elpd_loo · bar = marginal SE, text = paired SE of the "
                                    "difference", "Pareto-k diagnostic (4,445 points each)"])
for nm in order:
    l, col = loos[nm], BCOL[tuple(LEAD if nm == BNAME[tuple(LEAD)] else ALT)]
    fig.add_trace(go.Scatter(x=[l.elpd_loo], y=[nm], mode="markers", name=nm, legendgroup=nm,
                             marker=dict(color=col, size=13, symbol="diamond",
                                         line=dict(color="white", width=1)),
                             error_x=dict(type="data", array=[l.se], color=col,
                                          thickness=1.5, width=6),
                             hovertemplate="%{y}: %{x:.1f}<extra></extra>"), row=1, col=1)
    d_, se_ = CMP.loc[nm, "elpd_diff"], CMP.loc[nm, "dse"]
    fig.add_annotation(x=l.elpd_loo, y=nm, yshift=24, showarrow=False, row=1, col=1,
                       font=dict(size=10.5, color=C["dark"]),
                       text=f"{l.elpd_loo:.1f} ± {l.se:.1f}" +
                            ("" if d_ == 0 else f"   ·   −{d_:.1f} ± {se_:.1f} vs best"))
    k = l.pareto_k.values
    fig.add_trace(go.Bar(x=["k ≤ 0.5", "0.5 < k ≤ 0.7", "0.7 < k ≤ 1", "k > 1"],
                         y=[int((k <= .5).sum()), int(((k > .5) & (k <= .7)).sum()),
                            int(((k > .7) & (k <= 1)).sum()), int((k > 1).sum())],
                         name=nm, legendgroup=nm, showlegend=False, marker_color=col,
                         text=[int((k <= .5).sum()), int(((k > .5) & (k <= .7)).sum()),
                               int(((k > .7) & (k <= 1)).sum()), int((k > 1).sum())],
                         textposition="outside", textfont=dict(size=9.5),
                         hovertemplate="%{x}: %{y}<extra></extra>"), row=1, col=2)
fig.add_trace(go.Scatter(x=["0.7 < k ≤ 1"], y=[None], mode="markers",
                         marker=dict(color="rgba(0,0,0,0)"),
                         name="k > 0.7 means the PSIS estimate is unreliable at that point"),
              row=1, col=2)
fig.update_xaxes(title_text="elpd_loo (± SE)", row=1, col=1)
fig.update_yaxes(categoryorder="array", categoryarray=order[::-1], tickfont=dict(size=10),
                 row=1, col=1)
fig.update_xaxes(title_text="Pareto shape k", row=1, col=2)
fig.update_yaxes(title_text="observations", range=[0, 3350], row=1, col=2)
fig.update_layout(title="LOO-CV over the same 4,445 observations, one basin against the other",
                  height=420, width=1240, bargap=0.25, legend=dict(orientation="h", y=-0.28),
                  margin=dict(t=90, b=130, l=190))
show(fig, "09_loo")

print(CMP.to_string())
for nm in order:
    l = loos[nm]
    k = l.pareto_k.values
    print(f"{nm:24s} elpd_loo {l.elpd_loo:9.1f} ± {l.se:.1f} · p_loo {l.p_loo:7.1f} · "
          f"k>0.7 {int((k > 0.7).sum()):4d} / {k.size}")
d_ = float(CMP["elpd_diff"].max())
print(f"elpd difference {d_:.1f} ± {float(CMP['dse'].max()):.1f} nats = "
      f"{d_ / float(CMP['dse'].max()):.2f} SE, in favour of the basin that sits "
      f"{LP[LEAD].mean() - LP[ALT].mean():.1f} nats LOWER in log posterior density. "
      f"Not readable in either direction.")

/Users/yassineessifi/miniforge3/envs/pymc_env/lib/python3.11/site-packages/arviz/stats/stats.py:782: UserWarning: Estimated shape parameter of Pareto distribution is greater than 0.68 for one or more samples. You should consider using a more robust model, this is because importance sampling is less likely to work well if the marginal posterior and LOO posterior are very different. This is more likely to happen with a non-robust model and highly influential observations.
  warnings.warn(


/Users/yassineessifi/miniforge3/envs/pymc_env/lib/python3.11/site-packages/arviz/stats/stats.py:782: UserWarning: Estimated shape parameter of Pareto distribution is greater than 0.66 for one or more samples. You should consider using a more robust model, this is because importance sampling is less likely to work well if the marginal posterior and LOO posterior are very different. This is more likely to happen with a non-robust model and highly influential observations.
  warnings.warn(


                        rank     elpd_loo        p_loo  elpd_diff    weight         se        dse  warning scale
basin C · chains 1,4       0  6942.095748  1460.185032   0.000000  0.537907  71.506078   0.000000     True   log
basin A · chains 0,2,3     1  6936.825529  1461.811094   5.270219  0.462093  69.302391  22.283462     True   log
basin C · chains 1,4     elpd_loo    6942.1 ± 71.5 · p_loo  1460.2 · k>0.7  677 / 4445
basin A · chains 0,2,3   elpd_loo    6936.8 ± 69.3 · p_loo  1461.8 · k>0.7  693 / 4445
elpd difference 5.3 ± 22.3 nats = 0.24 SE, in favour of the basin that sits 13.6 nats LOWER in log posterior density. Not readable in either direction.


### Verdict
Ceilings: 3 of 98 gaps are inferred (SWE-Bench Verified, WMDP Biology, MMLU — scores flush against d), 95 are the prior, and the 2 fixed FrontierMath v1 walls sit 0.046 / 0.121 above any score on record, so they are imposed, not learned.
The noise gap therefore buys a plateau correction on saturated OLD benchmarks; it says nothing about the new ones.
Axis 2 is three basins at alignment 0.95, one question: GBAEval on commonsense (2.41 / 2.26) or on fluid (2.20). Chain 5 is a 0.91 neighbour, and folding it in at 0.90 is what turns eta r̂ 1.025 into 1.473.
Report the 0.95 leading basin, not the 0.90 merge, and not the pooled posterior.
Overall ordering is safe (Spearman +0.983 on the summed axes); per-axis-2 abilities are not (+0.700), and LOO cannot arbitrate (5.3 ± 22.3 nats).